# 01 — Data inspection
Pakistan Glacier Intelligence Explorer · Siachen Glacier

Checks what we actually downloaded **before** any processing:
1. Find all ITS_LIVE GeoTIFFs and the NISAR GOFF HDF5 file
2. Inspect ITS_LIVE raster metadata (CRS, resolution, bounds)
3. Confirm all 7 variables × 6 years are present
4. Inspect the NISAR HDF5 structure
5. **Check whether both datasets actually cover Siachen Glacier**
6. Save a file inventory

> Run this with the local `.venv` kernel (not Colab) — the data lives on `D:\`.

In [1]:
from pathlib import Path
import re
import rasterio
import h5py
import numpy as np
import pandas as pd
from pyproj import Transformer

PROJECT_DIR = Path(r"D:\Pakistan Glacier Intelligence Explorer")
ITS_LIVE_DIR = PROJECT_DIR / "data" / "its_live"
NISAR_DIR = PROJECT_DIR / "data" / "nisar"
OUTPUT_DIR = PROJECT_DIR / "results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Approximate Siachen Glacier bounding box (WGS84 lon/lat)
SIACHEN_BBOX = dict(lon_min=76.75, lon_max=77.30, lat_min=35.15, lat_max=35.70)
SIACHEN_POINTS = {            # rough reference points along the glacier
    "terminus":   (77.08, 35.24),
    "mid-glacier": (77.00, 35.45),
    "upper basin": (76.85, 35.62),
}

assert ITS_LIVE_DIR.exists(), f"Missing {ITS_LIVE_DIR} — are you running on the local kernel?"
assert NISAR_DIR.exists(), f"Missing {NISAR_DIR}"
print("Project directory:", PROJECT_DIR)

Project directory: D:\Pakistan Glacier Intelligence Explorer


## 1. Find all downloaded files

In [2]:
tif_files = sorted(ITS_LIVE_DIR.rglob("*.tif"))
h5_files = sorted(NISAR_DIR.rglob("*.h5"))
print("ITS_LIVE GeoTIFF files:", len(tif_files))
print("NISAR HDF5 files:     ", len(h5_files))
for f in h5_files:
    print("  ", f.name)

ITS_LIVE GeoTIFF files: 42
NISAR HDF5 files:      1
   NISAR_L2_PR_GOFF_005_135_D_071_006_2000_SH_20251119T134628_20251119T134703_20251201T134629_20251201T134704_X05010_N_F_J_001.h5


## 2. ITS_LIVE raster metadata

In [3]:
records = []
for f in tif_files:
    with rasterio.open(f) as src:
        records.append({
            "year": f.parent.name,
            "filename": f.name,
            "width": src.width, "height": src.height, "bands": src.count,
            "dtype": src.dtypes[0], "crs": str(src.crs),
            "res_x": src.res[0], "res_y": src.res[1],
            "left": src.bounds.left, "bottom": src.bounds.bottom,
            "right": src.bounds.right, "top": src.bounds.top,
            "nodata": src.nodata,
        })
raster_df = pd.DataFrame(records)
raster_df.to_csv(OUTPUT_DIR / "its_live_metadata.csv", index=False)

print("Unique CRS:        ", raster_df.crs.unique().tolist())
print("Unique resolutions:", sorted(set(zip(raster_df.res_x, raster_df.res_y))))
print("Unique grid shapes:", sorted(set(zip(raster_df.width, raster_df.height))))
same_grid = raster_df[["width","height","left","bottom","right","top"]].drop_duplicates().shape[0] == 1
print("All 42 rasters share one grid:", same_grid)
raster_df[["year","filename","width","height","dtype","crs","res_x","nodata"]].head(7)

Unique CRS:         ['ESRI:102027']
Unique resolutions: [(120.0, 120.0)]
Unique grid shapes: [(26968, 17806)]
All 42 rasters share one grid: True


,year,filename,width,height,dtype,crs,res_x,nodata
0,2017,ITS_LIVE_velocity_120m_RGI14A_2017_V02.1_count...,26968,17806,uint32,ESRI:102027,120.0,0.0
1,2017,ITS_LIVE_velocity_120m_RGI14A_2017_V02.1_v.tif,26968,17806,float32,ESRI:102027,120.0,-32767.0
2,2017,ITS_LIVE_velocity_120m_RGI14A_2017_V02.1_v_err...,26968,17806,uint16,ESRI:102027,120.0,32767.0
3,2017,ITS_LIVE_velocity_120m_RGI14A_2017_V02.1_vx.tif,26968,17806,float32,ESRI:102027,120.0,-32767.0
4,2017,ITS_LIVE_velocity_120m_RGI14A_2017_V02.1_vx_er...,26968,17806,uint16,ESRI:102027,120.0,32767.0
5,2017,ITS_LIVE_velocity_120m_RGI14A_2017_V02.1_vy.tif,26968,17806,float32,ESRI:102027,120.0,-32767.0
6,2017,ITS_LIVE_velocity_120m_RGI14A_2017_V02.1_vy_er...,26968,17806,uint16,ESRI:102027,120.0,32767.0


## 3. Completeness check — 7 variables × 2017–2022

In [4]:
VARIABLES = ["v", "vx", "vy", "v_error", "vx_error", "vy_error", "count"]
pattern = re.compile(r"_(\d{4})_V\d+\.\d+_(\w+)$")

found = {}
odd_names = []
for f in tif_files:
    m = pattern.search(f.stem)
    if m:
        found.setdefault(int(m.group(1)), set()).add(m.group(2))
    else:
        odd_names.append(f.name)

table = pd.DataFrame(
    {year: ["FOUND" if v in found.get(year, set()) else "MISSING" for v in VARIABLES]
     for year in range(2017, 2023)},
    index=VARIABLES,
)
print("Files with unexpected names:", odd_names or "none")
table

Files with unexpected names: none


,2017,2018,2019,2020,2021,2022
v,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND
vx,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND
vy,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND
v_error,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND
vx_error,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND
vy_error,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND
count,FOUND,FOUND,FOUND,FOUND,FOUND,FOUND


## 4. NISAR GOFF HDF5 structure

In [5]:
nisar_file = h5_files[0]
datasets = []
with h5py.File(nisar_file, "r") as f:
    def visit(name, obj):
        if isinstance(obj, h5py.Dataset):
            datasets.append({"path": name, "shape": obj.shape, "dtype": str(obj.dtype)})
    f.visititems(visit)
ds_df = pd.DataFrame(datasets)
ds_df.to_csv(OUTPUT_DIR / "nisar_structure.csv", index=False)
print("Total datasets in file:", len(ds_df))

# The large 2-D layers are the actual measurements
big = ds_df[ds_df["shape"].apply(lambda s: len(s) == 2 and min(s) > 100)]
print("\n2-D measurement layers:")
for _, r in big.iterrows():
    print(f"  {r['path']}  {r['shape']}  {r['dtype']}")

Total datasets in file: 194

2-D measurement layers:
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1/alongTrackOffset  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1/alongTrackOffsetVariance  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1/correlationSurfacePeak  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1/crossOffsetVariance  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1/slantRangeOffset  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1/slantRangeOffsetVariance  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer1/snr  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer2/alongTrackOffset  (4302, 4365)  float32
  science/LSAR/GOFF/grids/frequencyA/pixelOffsets/HH/layer2/alongTrackOffsetVariance  (4302, 4365)  float32
  science/LSAR/GOFF/grids/f

In [6]:
# Key identification metadata (acquisition dates, orbit, look direction, polygon)
def read_val(ds):
    v = ds[()]
    if isinstance(v, bytes):
        return v.decode(errors="replace")
    if isinstance(v, np.ndarray) and v.dtype.kind == "S":
        return [x.decode(errors="replace") for x in v.ravel()]
    return v

ident = {}
with h5py.File(nisar_file, "r") as f:
    for p in ds_df.path:
        if "/identification/" in p and f[p].size < 50:
            ident[p.split("/")[-1]] = read_val(f[p])
for k in sorted(ident):
    val = str(ident[k])
    print(f"{k:40s} {val[:110]}")

boundingPolygon                          POLYGON ((74.8932839211468 36.1301770675886 3655.51673060657,74.806743017912 35.921286991793 2999.29443355673,
compositeReleaseId                       X05010
diagnosticModeFlag                       0
frameNumber                              71
granuleId                                NISAR_L2_PR_GOFF_005_135_D_071_006_2000_SH_20251119T134628_20251119T134703_20251201T134629_20251201T134704_X05
instrumentName                           L-SAR
isDithered                               False
isFullFrame                              True
isGeocoded                               True
isMixedMode                              False
isUrgentObservation                      False
listOfFrequencies                        ['A']
lookDirection                            Left
missionId                                NISAR
orbitPassDirection                       Descending
platformName                             NISAR
processingCenter                         J

## 5. Does our data actually cover Siachen?

In [7]:
def in_bbox(lon, lat, b=SIACHEN_BBOX):
    return b["lon_min"] <= lon <= b["lon_max"] and b["lat_min"] <= lat <= b["lat_max"]

def point_in_poly(x, y, poly):
    inside = False
    for (x1, y1), (x2, y2) in zip(poly, poly[1:] + poly[:1]):
        if (y1 > y) != (y2 > y) and x < (x2 - x1) * (y - y1) / (y2 - y1) + x1:
            inside = not inside
    return inside

# --- NISAR footprint from its boundingPolygon (WKT, lon/lat) ---
poly_wkt = ident.get("boundingPolygon")
nisar_poly = [tuple(map(float, p.split()[:2]))
              for p in re.findall(r"-?\d+\.?\d*\s+-?\d+\.?\d*(?:\s+-?\d+\.?\d*)?", str(poly_wkt))]
lons, lats = zip(*nisar_poly)
print(f"NISAR footprint: lon {min(lons):.3f} → {max(lons):.3f}, lat {min(lats):.3f} → {max(lats):.3f}")
for name, (lon, lat) in SIACHEN_POINTS.items():
    print(f"  Siachen {name:12s} ({lon}, {lat}) inside NISAR footprint: {point_in_poly(lon, lat, nisar_poly)}")

# --- ITS_LIVE grid extent, converted to lon/lat ---
r0 = raster_df.iloc[0]
to_ll = Transformer.from_crs(r0.crs, "EPSG:4326", always_xy=True)
to_xy = Transformer.from_crs("EPSG:4326", r0.crs, always_xy=True)
print(f"\nITS_LIVE grid CRS: {r0.crs}")
for name, (lon, lat) in SIACHEN_POINTS.items():
    x, y = to_xy.transform(lon, lat)
    inside = r0.left <= x <= r0.right and r0.bottom <= y <= r0.top
    print(f"  Siachen {name:12s} inside ITS_LIVE grid: {inside}")

NISAR footprint: lon 74.134 → 77.597, lat 33.348 → 36.130
  Siachen terminus     (77.08, 35.24) inside NISAR footprint: True
  Siachen mid-glacier  (77.0, 35.45) inside NISAR footprint: True
  Siachen upper basin  (76.85, 35.62) inside NISAR footprint: True

ITS_LIVE grid CRS: ESRI:102027
  Siachen terminus     inside ITS_LIVE grid: True
  Siachen mid-glacier  inside ITS_LIVE grid: True
  Siachen upper basin  inside ITS_LIVE grid: True


In [8]:
# Do ITS_LIVE pixels over Siachen hold real velocity values? (reads only a small window)
from rasterio.windows import from_bounds
b = SIACHEN_BBOX
rows = []
for year in range(2017, 2023):
    vf = next(f for f in tif_files if f.stem.endswith(f"_{year}_V02.1_v"))
    with rasterio.open(vf) as src:
        xs, ys = to_xy.transform([b["lon_min"], b["lon_max"], b["lon_min"], b["lon_max"]],
                                 [b["lat_min"], b["lat_min"], b["lat_max"], b["lat_max"]])
        win = from_bounds(min(xs), min(ys), max(xs), max(ys), src.transform)
        arr = src.read(1, window=win, masked=True).astype("float64")
        valid = arr.compressed()
        valid = valid[np.isfinite(valid)]
        rows.append({"year": year, "window_px": arr.size, "valid_px": valid.size,
                     "valid_%": round(100 * valid.size / arr.size, 1),
                     "median_v_m_per_yr": round(float(np.median(valid)), 1) if valid.size else None,
                     "p95_v_m_per_yr": round(float(np.percentile(valid, 95)), 1) if valid.size else None})
siachen_df = pd.DataFrame(rows)
siachen_df.to_csv(OUTPUT_DIR / "its_live_siachen_window_stats.csv", index=False)
siachen_df

,year,window_px,valid_px,valid_%,median_v_m_per_yr,p95_v_m_per_yr
0,2017,247977,247960,100.0,2.6,111.5
1,2018,247977,247977,100.0,2.2,116.9
2,2019,247977,247969,100.0,2.0,113.7
3,2020,247977,247100,99.6,2.1,110.7
4,2021,247977,247977,100.0,2.3,130.7
5,2022,247977,247977,100.0,1.4,125.7


## 6. File inventory

In [9]:
inventory = pd.DataFrame([{
    "filename": f.name, "folder": f.parent.name, "file_type": f.suffix,
    "size_MB": round(f.stat().st_size / 1024**2, 2), "path": str(f)}
    for f in tif_files + h5_files])
inventory.to_csv(OUTPUT_DIR / "project_file_inventory.csv", index=False)
print("Total files:", len(inventory), "| Total size (GB):", round(inventory.size_MB.sum() / 1024, 2))
inventory.groupby("folder").size_MB.sum().round(1)

Total files: 43 | Total size (GB): 15.34


folder
2017     2457.1
2018     2461.3
2019     2452.6
2020     2462.2
2021     2487.9
2022     2514.3
nisar     872.0
Name: size_MB, dtype: float64